In [ ]:
!wget --no-cache -O init.py -q https://raw.githubusercontent.com/UDEA-Esp-Analitica-y-Ciencia-de-Datos/EACD-08-CLOUD/master/init.py
import init; init.init(force_download=False);
from IPython.display import Image

In [ ]:
Image("local/imgs/unidad-2.png")


# Batch Processing
## Simulación de *ventas diarias* y generación de reporte **batch**

Vamos a comprender de forma práctica, cómo funciona el **Batch Processing** usando un escenario cotidiano:

> **Una tienda registra ventas durante el día** y al final genera un **reporte consolidado** (batch).

### ¿Qué aprenderás aquí?
- Qué significa **acumular datos** antes de procesarlos.
- Por qué el batch tiene **latencia alta** y por qué, en muchos casos, eso es **aceptable**.
- Cómo un proceso batch produce **resultados agregados** (totales, promedios, top productos) en vez de reaccionar evento a evento.


## 1. Conceptos básicos antes de empezar
Antes de ver código, aclaremos algunas ideas que usaremos en este notebook.
### 🧱 ¿Qué es Batch Processing?
Batch Processing es un paradigma donde:
1. **Los datos se generan y almacenan** durante un período (por ejemplo, un día).
2. Luego, en un momento específico (por ejemplo, a las 11:59 pm), se ejecuta un **proceso batch**.
3. El proceso batch produce resultados **consolidados**: totales, promedios, rankings, tablas resumen.

### ⏳ ¿Por qué la latencia es aceptable?
Porque en escenarios como:
- Reportes diarios de ventas
- Facturación
- Nómina
- Análisis histórico

no necesitamos actuar en milisegundos. Lo importante es:
- **Exactitud**
- **Consistencia**
- **Completitud**

### 🎯 ¿Cuál es el “producto” típico de un batch?
Un **reporte** (o tablas agregadas) que responde a preguntas como:
- ¿Cuánto vendimos hoy?
- ¿Cuál fue el producto más vendido?
- ¿En qué hora vendimos más?
- ¿Cuál fue el ticket promedio?

---

En este notebook simularemos un día de ventas (acelerado) y luego generaremos ese reporte.



## 2. Preparar el entorno

Usaremos:
- `pandas` para manipular datos tabulares.
- `matplotlib` para gráficos.
- `datetime` para estampas de tiempo

In [ ]:
import random
import math
from datetime import datetime, timedelta, timezone

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrow



## 3. Escenario: Tienda con catálogo de productos
Vamos a simular una tienda con un catálogo sencillo. Cada venta tendrá:

- Fecha/hora
- Producto
- Cantidad
- Precio unitario
- Total de la venta

En Batch Processing, el flujo típico es:

1. **Se registran ventas** durante el día (se guardan en almacenamiento).
2. **No se procesan** una por una para producir métricas en tiempo real.
3. Al final del día se ejecuta un **proceso batch** que genera el reporte.


In [ ]:
Image("local/imgs/tienda.png")

In [ ]:
CATALOGO = [
    {"producto_id": "P001", "nombre": "Café",     "precio": 2.5},
    {"producto_id": "P002", "nombre": "Pan",      "precio": 1.2},
    {"producto_id": "P003", "nombre": "Leche",    "precio": 1.8},
    {"producto_id": "P004", "nombre": "Huevos",   "precio": 3.9},
    {"producto_id": "P005", "nombre": "Arroz",    "precio": 2.1},
    {"producto_id": "P006", "nombre": "Queso",    "precio": 4.5},
    {"producto_id": "P007", "nombre": "Galletas", "precio": 1.6},
    {"producto_id": "P008", "nombre": "Jugo",     "precio": 2.9},
]

df_catalogo = pd.DataFrame(CATALOGO)
df_catalogo


## 4. Simular ventas durante un día (datos que se acumulan)

Generaremos ventas con timestamps a lo largo del día y con “horas pico” (mayor cantidad de ventas en ciertos horarios).

**Importante:** en este punto solo *generamos y almacenamos* los datos.  
Aún no generamos reportes (eso vendrá al final).


In [ ]:
def utc_iso(dt: datetime) -> str:
    """Devuelve un timestamp ISO 8601 en UTC, terminado en Z."""
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")

def peso_hora(h: int) -> float:
    """Función simple para simular horas pico (más ventas en ciertas horas)."""
    pico_1 = math.exp(-((h - 9) ** 2) / (2 * 1.5 ** 2))   # ~9am
    pico_2 = math.exp(-((h - 18) ** 2) / (2 * 1.8 ** 2))  # ~6pm
    base = 0.15
    return base + pico_1 + pico_2

def simular_ventas_diarias(
    fecha_local: datetime,
    num_ventas: int = 1200,
    tz=timezone.utc,
    seed: int = 42
) -> pd.DataFrame:
    """Simula ventas de un solo día con mayor probabilidad en horas pico."""
    random.seed(seed)

    inicio_dia = datetime(fecha_local.year, fecha_local.month, fecha_local.day, 0, 0, 0, tzinfo=tz)

    # distribución por hora
    pesos = [peso_hora(h) for h in range(24)]
    suma = sum(pesos)
    probs = [p / suma for p in pesos]

    ventas = []
    for venta_id in range(1, num_ventas + 1):
        hora = random.choices(range(24), weights=probs, k=1)[0]
        minuto = random.randint(0, 59)
        segundo = random.randint(0, 59)
        ts = datetime(inicio_dia.year, inicio_dia.month, inicio_dia.day, hora, minuto, segundo, tzinfo=tz)

        item = random.choice(CATALOGO)
        cantidad = random.choices([1, 2, 3, 4, 5], weights=[55, 25, 12, 6, 2], k=1)[0]
        precio = float(item["precio"])
        total = round(cantidad * precio, 2)

        ventas.append({
            "venta_id": venta_id,
            "timestamp": ts,
            "timestamp_iso": utc_iso(ts),
            "producto_id": item["producto_id"],
            "producto": item["nombre"],
            "cantidad": cantidad,
            "precio_unitario": precio,
            "total": total,
        })

    df = pd.DataFrame(ventas).sort_values("timestamp").reset_index(drop=True)
    return df

df_ventas = simular_ventas_diarias(datetime.now(timezone.utc), num_ventas=1200, tz=timezone.utc, seed=42)
df_ventas.head(10)


### 4.1 “Almacenamiento” de las ventas (persistencia)

En un sistema real, estas ventas se guardarían en una base de datos o en un data lake.  
Aquí las guardaremos como un CSV para simular persistencia.


In [ ]:
ruta_csv = "local/data/ventas_diarias.csv"
df_ventas.to_csv(ruta_csv, index=False)
ruta_csv


✅ Hasta aquí, lo único que hicimos fue:
- Generar ventas
- Almacenarlas (CSV).

**Todavía no hicimos ninguna métrica final.**  
Eso es intencional: en batch, el procesamiento ocurre al final.



## 5. Paso “batch”: generar el reporte diario

Ahora simularemos el momento del batch:

> “Terminó el día. Ejecutemos el job batch del reporte diario.”

El job leerá el CSV y construirá resultados agregados como:
- Ventas totales
- Ingresos totales
- Ticket promedio
- Top productos
- Ventas por hora


In [ ]:
def job_batch_reporte_diario(ruta_csv: str) -> dict:
    df = pd.read_csv(ruta_csv, parse_dates=["timestamp"])
    df["hora"] = df["timestamp"].dt.hour

    ventas_totales = int(df["venta_id"].nunique())
    ingresos_totales = float(df["total"].sum())
    unidades_totales = int(df["cantidad"].sum())
    ticket_promedio = float(df["total"].mean())

    top_ingresos = (
        df.groupby(["producto_id", "producto"], as_index=False)["total"]
        .sum()
        .sort_values("total", ascending=False)
        .head(5)
        .reset_index(drop=True)
    )

    top_unidades = (
        df.groupby(["producto_id", "producto"], as_index=False)["cantidad"]
        .sum()
        .sort_values("cantidad", ascending=False)
        .head(5)
        .reset_index(drop=True)
    )

    ventas_por_hora = (
        df.groupby("hora", as_index=False)
        .agg(ventas=("venta_id", "count"), ingresos=("total", "sum"), unidades=("cantidad", "sum"))
        .sort_values("hora")
        .reset_index(drop=True)
    )

    return {
        "df": df,
        "ventas_totales": ventas_totales,
        "ingresos_totales": ingresos_totales,
        "unidades_totales": unidades_totales,
        "ticket_promedio": ticket_promedio,
        "top_ingresos": top_ingresos,
        "top_unidades": top_unidades,
        "ventas_por_hora": ventas_por_hora,
    }

reporte = job_batch_reporte_diario(ruta_csv)
reporte["ventas_totales"], reporte["ingresos_totales"], reporte["unidades_totales"], reporte["ticket_promedio"]


### 5.1 Resultados principales del reporte batch

Este es el tipo de salida que esperarías en un reporte diario.  
Observa que todo esto se calcula **después** de que el día terminó.


In [ ]:
print("=== REPORTE BATCH DIARIO ===")
print(f"Ventas totales:   {reporte['ventas_totales']}")
print(f"Ingresos totales: {reporte['ingresos_totales']:.2f}")
print(f"Unidades totales: {reporte['unidades_totales']}")
print(f"Ticket promedio:  {reporte['ticket_promedio']:.2f}")


### 5.2 Top productos (por ingresos)


In [ ]:
reporte["top_ingresos"]


### 5.3 Top productos (por unidades)


In [ ]:
reporte["top_unidades"]


## 6. Visualizaciones típicas de un reporte batch


In [ ]:
ventas_por_hora = reporte["ventas_por_hora"]

plt.figure(figsize=(10, 4))
plt.plot(ventas_por_hora["hora"], ventas_por_hora["ventas"], marker="o")
plt.title("Ventas por hora (consolidado batch)")
plt.xlabel("Hora del día")
plt.ylabel("Número de ventas")
plt.xticks(range(0, 24, 1))
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ventas_por_hora["hora"], ventas_por_hora["ingresos"], marker="o")
plt.title("Ingresos por hora (consolidado batch)")
plt.xlabel("Hora del día")
plt.ylabel("Ingresos")
plt.xticks(range(0, 24, 1))
plt.grid(True)
plt.show()


## 7. ¿Dónde aparece la “latencia” en batch?

En batch, una venta que ocurre a las 9:15 am puede verse reflejada en el reporte final muchas horas después.

Esto es aceptable cuando:
- el objetivo es consolidar el día,
- el negocio toma decisiones con reportes diarios,
- se prioriza consistencia y costo.

No sería aceptable en casos como fraude o alertas críticas (eso es streaming).



## 8. Ejercicios propuestos

1. Cambia `num_ventas` a 5000 o 20000.  
2. Agrega una columna `descuento` y calcula ingresos netos.  
3. Segmenta ingresos por franjas horarias (mañana/tarde/noche).  
4. Calcula top productos por franja horaria.  
5. Introduce datos erróneos (cantidad 0, precio negativo) y crea una etapa de limpieza antes del reporte.
